In [4]:
import cv2
import mediapipe as mp
import pyautogui
import time

pyautogui.PAUSE = 0

class SubwayController:
    def __init__(self):
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.8)
        
        # State tracking for the 3-Lane Switch
        self.virtual_lane = 0 # -1: Left, 0: Center, 1: Right
        self.last_y_action = 0
        self.cooldown = 0.3

    def process_frame(self, frame):
        h, w, _ = frame.shape
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.hands.process(img_rgb)
        
        # UI: Draw a vertical "Neutral" guide in the box
        cv2.rectangle(frame, (w//2-150, h//2-150), (w//2+150, h//2+150), (255, 255, 255), 1)

        if results.multi_hand_landmarks:
            lmList = []
            for id, lm in enumerate(results.multi_hand_landmarks[0].landmark):
                lmList.append([id, int(lm.x * w), int(lm.y * h)])

            # Wrist (0) and Middle Finger Base (9) define the TILT
            wrist_x = lmList[0][1]
            palm_x = lmList[9][1]
            
            # --- LANE TOGGLE LOGIC ---
            tilt_threshold = 35 # Sensitivity of the tilt
            tilt_val = palm_x - wrist_x
            
            # Identify which "State" the hand is in
            if tilt_val < -tilt_threshold:
                target_lane = -1 # Physical Left
            elif tilt_val > tilt_threshold:
                target_lane = 1  # Physical Right
            else:
                target_lane = 0  # Physical Center

            # --- KEYPRESS LOGIC (Triggered by state change) ---
            if target_lane != self.virtual_lane:
                if target_lane < self.virtual_lane:
                    # If we move hand towards the left (1->0 or 0->-1)
                    pyautogui.press('left')
                elif target_lane > self.virtual_lane:
                    # If we move hand towards the right (-1->0 or 0->1)
                    pyautogui.press('right')
                
                self.virtual_lane = target_lane

            # --- FINGER GESTURES (Jump/Roll) ---
            # Index Tip (8) vs Index PIP (6)
            index_up = lmList[8][2] < lmList[6][2]
            middle_up = lmList[12][2] < lmList[10][2]
            ring_up = lmList[16][2] < lmList[14][2]

            curr_time = time.time()
            if curr_time - self.last_y_action > self.cooldown:
                # FIST to Roll (Index, Middle, Ring are all down)
                if not index_up and not middle_up and not ring_up:
                    pyautogui.press('down')
                    self.last_y_action = curr_time
                # INDEX to Jump (Index up, Middle down)
                elif index_up and not middle_up:
                    pyautogui.press('up')
                    self.last_y_action = curr_time

            # Visual Indicator
            color = (0, 255, 0) if target_lane == 0 else (255, 0, 0)
            cv2.line(frame, (lmList[0][1], lmList[0][2]), (lmList[9][1], lmList[9][2]), color, 5)

        return frame

def main():
    cap = cv2.VideoCapture(0)
    controller = SubwayController()
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        frame = cv2.flip(frame, 1)
        frame = controller.process_frame(frame)
        cv2.imshow("Subway 3-Lane Tilt Control", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

In [4]:
import mediapipe as mp
try:
    print(f"MediaPipe version: {mp.__version__}")
    print(f"Solutions found: {mp.solutions}")
    print("SUCCESS: MediaPipe is working correctly!")
except AttributeError:
    print("FAILURE: MediaPipe still cannot find 'solutions'.")

MediaPipe version: 0.10.31
FAILURE: MediaPipe still cannot find 'solutions'.
